# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [110]:
#@title 1.1 — Install dependencies

%pip install -q \
    neo4j \
    pandas \
    numpy \
    pyarrow \
    sentence-transformers \
    faiss-cpu \
    groq \
    openai \
    tqdm \
    networkx \
    spacy \
    datasets \
    python-dotenv

In [69]:
import numpy as np
import pandas as pd
import faiss

from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from groq import Groq

print("✅ Core dependencies OK")

✅ Core dependencies OK


In [131]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

DATA_PATH = "/content/drive/MyDrive/hackernoon_subset.csv"
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

In [127]:
print("Neo4j:", bool(NEO4J_URI))
print("Neo4j password:", bool(NEO4J_PASSWORD))
print("Groq:", bool(GROQ_API_KEY))
print("Groq model:", GROQ_MODEL)
print("HF:", bool(HF_TOKEN))
print("Judge:", JUDGE_PROVIDER, JUDGE_MODEL)

Neo4j: True
Neo4j password: True
Groq: True
Groq model: openai/gpt-oss-120b
HF: True
Judge: groq openai/gpt-oss-120b


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [19]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = "/content/drive/MyDrive/hackernoon_subset.csv"

# Giới hạn cho bản lab. Có thể tăng sau buổi học.
LIMIT_ROWS = 5000
LIMIT_MB = 300

# True  -> progress/dừng ưu tiên theo MB
# False -> progress theo rows; vẫn có hard-stop LIMIT_ROWS
PRIORITIZE_MB = True

# Đọc từ Colab Secrets qua get_secret() ở cell config.
if not HF_TOKEN:
    raise ValueError(
        "Thiếu HF_TOKEN. Hãy thêm Hugging Face Access Token vào Colab Secrets với tên HF_TOKEN."
    )

print("Đang kết nối luồng dữ liệu (streaming)...")

try:
    dataset = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    iterator = iter(dataset)

    first_row = next(iterator)
    headers = list(first_row.keys())

    print(f"Đang ghi dữ liệu vào: {OUTPUT_CSV}")

    rows_written = 0
    total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
    unit_progress = "MB" if PRIORITIZE_MB else "row"

    with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        writer.writerow(first_row)
        rows_written += 1

        # Flush để kích thước file phản ánh dữ liệu vừa ghi.
        f.flush()
        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

        with tqdm(
            total=total_progress,
            desc=f"Đang tải ({unit_progress})",
            unit=unit_progress,
        ) as pbar:
            if PRIORITIZE_MB:
                pbar.n = min(file_size_mb, LIMIT_MB)
                pbar.refresh()
            else:
                pbar.update(1)

            for row in iterator:
                writer.writerow(row)
                rows_written += 1

                # Kiểm tra dung lượng định kỳ để giảm overhead I/O.
                # Khi gần LIMIT_MB, kiểm tra mỗi row để dừng sát ngưỡng hơn.
                should_check_size = (
                    PRIORITIZE_MB
                    and (
                        rows_written % 100 == 0
                        or file_size_mb >= LIMIT_MB * 0.95
                    )
                )

                if should_check_size:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                    pbar.refresh()
                elif not PRIORITIZE_MB:
                    pbar.update(1)

                # Hard-stop theo MB nếu đang ưu tiên dung lượng.
                if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn dung lượng: "
                        f"{file_size_mb:.2f} MB "
                        f"(Tổng: {rows_written:,} dòng)"
                    )
                    break

                # Hard-stop theo số dòng trong mọi chế độ.
                if rows_written >= LIMIT_ROWS:
                    f.flush()
                    file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                    print(
                        f"\n[DỪNG] Đã đạt giới hạn số dòng: "
                        f"{rows_written:,} dòng "
                        f"(Dung lượng: {file_size_mb:.2f} MB)"
                    )
                    break

        f.flush()

    final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
    print(
        f"✅ Hoàn thành: {os.path.abspath(OUTPUT_CSV)}\n"
        f"   Rows: {rows_written:,}\n"
        f"   Size: {final_size_mb:.2f} MB"
    )

    # Đồng bộ đường dẫn cho cell loader tiếp theo.
    DATA_PATH = OUTPUT_CSV

except StopIteration:
    raise RuntimeError("Dataset stream rỗng: không lấy được dòng đầu tiên.")
except Exception as e:
    print(f"\n❌ Có lỗi xảy ra: {e}")
    print(
        "Kiểm tra: (1) HF_TOKEN, (2) quyền Agree/Access trên Hugging Face, "
        "(3) kết nối mạng của Colab."
    )
    raise

Đang kết nối luồng dữ liệu (streaming)...
Đang ghi dữ liệu vào: /content/drive/MyDrive/hackernoon_subset.csv


Đang tải (MB):   0%|          | 0/300 [00:00<?, ?MB/s]


[DỪNG] Đã đạt giới hạn số dòng: 5,000 dòng (Dung lượng: 2.92 MB)
✅ Hoàn thành: /content/drive/MyDrive/hackernoon_subset.csv
   Rows: 5,000
   Size: 2.92 MB


In [18]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()

✅ Neo4j connected.
✅ Schema ready.


In [17]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    text_col = pick_col(
    raw,
    ["text", "content", "article", "body", "story", "description"]
)
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.sample(LAB_MAX_ARTICLES, random_state=SEED).sort_index().reset_index(drop=True)
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
display(chunks_df.head())

Exact dedup: 2,675 -> 2,105


Chunking:   0%|          | 0/1500 [00:00<?, ?it/s]

,chunk_id,article_id,title,published_date,text
0,1a05beb7aa3071be6fd7::c0000,1a05beb7aa3071be6fd7,onsemi and Sineng Electric Spearhead the Development of Sustainable Energy Applications,2023-05-16,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...
1,8e922bc62b578e73e815::c0000,8e922bc62b578e73e815,Modernizing State Services: Harnessing Technology for Enhanced Public Service Delivery,2023-05-01,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...
2,4bd7afdba71243b0dbcd::c0000,4bd7afdba71243b0dbcd,Terry Richardson On Why He Left AMD GreenPages’ Technology Chops And The AI Opportunity,2023-05-02,In February GreenPages acquired Toronto-based Zanaris an IT automation cloud and DevOps services firm ... Steve Burk...
3,6633c15d86f5e81f47b8::c0000,6633c15d86f5e81f47b8,5 Kubernetes technology vendors hot right now,2023-02-28,Kubernetes is a technology that has created a whole new ecosystem around itself and it is now a key plank in the Dev...
4,4ce72a4490a6a618e2d5::c0000,4ce72a4490a6a618e2d5,Bachelor of Science in Health Information Management,2023-08-16,Health information management (HIM) is a diverse yet evolving field that incorporates medicine management finance in...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [132]:
#@title 1.6 — LLM wrapper có retry + JSON parsing
from groq import Groq
import re
import json
import time
import random

groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    model = model or GROQ_MODEL
    if not model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}

            resp = groq_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            msg = str(e)
            # Quota token/day -> retry ngay cũng vô ích
            if (
                "429" in msg
                or "tokens per day" in msg.lower()
                or "rate_limit_exceeded" in msg.lower()
            ):
                raise RuntimeError(e)

            if attempt == max_retries - 1:
                break

            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    model = model or GROQ_MODEL
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    try:
        text, usage = groq_chat(messages, model=model, json_mode=True)
        return parse_json_object(text), usage
    except Exception as e:
        msg = str(e)
        if "json_validate_failed" not in msg and "Failed to validate JSON" not in msg:
            raise
        print("⚠️ JSON validation failed. Retry without response_format.")

    fallback_messages = [
        {
            "role": "system",
            "content": system + "\n\nReturn exactly ONE valid JSON object. No markdown. No code fence. No explanation. Do not call tools."
        },
        {"role": "user", "content": user},
    ]
    text, usage = groq_chat(fallback_messages, model=model, json_mode=False)
    return parse_json_object(text), usage

## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [11]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = groq_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=5):
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
    return pd.concat(out, ignore_index=True)

extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df = run_coref(extraction_source)
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")

Coref:   0%|          | 0/80 [00:00<?, ?it/s]

⚠️ JSON validation failed. Retry without response_format.
⚠️ JSON validation failed. Retry without response_format.
⚠️ JSON validation failed. Retry without response_format.
⚠️ JSON validation failed. Retry without response_format.


In [24]:
# Lấy các chunk coref fail
failed_mask = (
    extraction_source["unresolved_mentions"]
    .astype(str)
    .str.contains("COREF_BATCH_FAILED", na=False)
)

failed_chunks = extraction_source.loc[
    failed_mask,
    ["chunk_id", "text"]
].copy()

print("Chunks cần retry:", len(failed_chunks))
display(failed_chunks.head())

Chunks cần retry: 60


,chunk_id,text
165,50cfb7c09f27bb89f754::c0000,The new AM metal label allows retailers to protect previously unprotectable items reducing some of the risks associa...
166,e8a1bb5de5304ab2bc13::c0000,The Community College of Allegheny County is offering free computer information technology courses in September. Acc...
167,24c46dfb3f11cfdb89fe::c0000,Digital Services Act (DSA) The DSA is a groundbreaking ... with 70 sites “hiding” important information such as deli...
168,b4b23faeb8022a4d9762::c0000,This lack of access to technology creates a digital divide which can limit the ability of people to access informati...
169,8dc0de1ac5eca8c4bb72::c0000,The company’s multiple products were then centralised in a single Akeneo product information ... services company ha...


In [25]:
def resolve_coref_single(row):
    prompt = f"""
Resolve coreferences in this text conservatively.

Rules:
- Only resolve pronouns/references when antecedent is explicit.
- Never invent entities or facts.
- Preserve dates, numbers, tickers and product names.
- If uncertain, leave the mention unchanged.

Return exactly one JSON object:
{{
  "chunk_id": "{row.chunk_id}",
  "resolved_text": "...",
  "unresolved_mentions": []
}}

TEXT:
{row.text}
""".strip()

    obj, usage = groq_json(
        COREF_SYSTEM,
        prompt
    )

    return {
        "chunk_id": row.chunk_id,
        "resolved_text": norm_space(
            obj.get("resolved_text") or row.text
        ),
        "unresolved_mentions":
            obj.get("unresolved_mentions", []),
    }

In [33]:
retry_rows = []
retry_errors = []

for r in tqdm(
    failed_chunks.itertuples(index=False),
    total=len(failed_chunks),
    desc="Retry failed coref"
):
    try:
        retry_rows.append(
            resolve_coref_single(r)
        )

    except Exception as e:
        retry_errors.append({
            "chunk_id": r.chunk_id,
            "error": str(e)
        })

        # fallback giữ text gốc
        retry_rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": r.text,
            "unresolved_mentions": [
                "COREF_SINGLE_FAILED"
            ],
        })

Retry failed coref:   0%|          | 0/60 [00:00<?, ?it/s]

# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [39]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt)

def run_extraction(source_df, batch_size=4):
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    triples, errors = [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            errors.append({"start": start, "error": str(e)})
            continue

        for item in obj.get("items", []):
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            for x in item.get("relations", []):
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": float(x.get("confidence") or 0.0),
                })

    TRIPLE_COLUMNS = [
    "source_raw",
    "source_type",
    "relation",
    "target_raw",
    "target_type",
    "source_chunk_id",
    "published_date",
    "evidence",
    "confidence",
]
    ERROR_COLUMNS = [
    "start",
    "error",
]
    triples_df = pd.DataFrame(triples, columns=TRIPLE_COLUMNS)
    errors_df = pd.DataFrame(errors, columns=ERROR_COLUMNS)
    return triples_df, errors_df
raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
display(raw_triples_df.head())

NER+RE:   0%|          | 0/100 [00:00<?, ?it/s]

,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Sineng Electric,Company,USES,EliteSiC,Technology,1a05beb7aa3071be6fd7::c0000,2023-05-16,Sineng Electric will integrate onsemi EliteSiC silic,1.00
1,GreenPages,Company,ACQUIRED,Zanaris,Company,4bd7afdba71243b0dbcd::c0000,2023-05-02,GreenPages acquired Toronto-based Zanaris,1.00
2,Richard Jamieson,Person,WORKED_AT,Press Gazette,Company,5f54d2695696a7b61748::c0000,2022-12-27,Contact Richard Jamieson to enquire about working with Press Gazette,0.70
3,Mark Leary,Person,WORKED_AT,IDC,Company,24e2333c327a81eca0f6::c0000,2023-03-03,Mark Leary research director for network analytics and automation at IDC,1.00
4,dynaCERT,Company,DEVELOPED,HydraGEN™ Carbon Emission Reduction Technology,Technology,7ac7c9c97aac23e234ea::c0000,2023-09-18,dynaCERT’s HydraGEN™ Carbon Emission Reduction Technology line of commercial products,0.95


In [40]:
for v in [
    "chunks_df",
    "extraction_source",
    "raw_triples_df",
]:
    print(v, v in globals())

chunks_df True
extraction_source True
raw_triples_df True


In [41]:
test_source = extraction_source.head(4).copy()

print("Test chunks:")
display(
    test_source[
        ["chunk_id", "published_date", "resolved_text"]
        if "resolved_text" in test_source.columns
        else ["chunk_id", "published_date", "text"]
    ]
)

Test chunks:


,chunk_id,published_date,resolved_text
0,1a05beb7aa3071be6fd7::c0000,2023-05-16,(Nasdaq: ON) a leader in intelligent power and sensing technologies today announced that Sineng Electric will integr...
1,8e922bc62b578e73e815::c0000,2023-05-01,To deliver 21st-century government services Governors and cabinet members need leaders with technology expertise to ...
2,4bd7afdba71243b0dbcd::c0000,2023-05-02,In February GreenPages acquired Toronto-based Zanaris an IT automation cloud and DevOps services firm ... Steve Burk...
3,6633c15d86f5e81f47b8::c0000,2023-02-28,Kubernetes is a technology that has created a whole new ecosystem around itself and Kubernetes is now a key plank in...


In [43]:
print("shape:", raw_triples_df.shape)
print("columns:", raw_triples_df.columns.tolist())

required_cols = {
    "source_raw",
    "source_type",
    "relation",
    "target_raw",
    "target_type",
    "source_chunk_id",
    "published_date",
    "evidence",
    "confidence",
}

missing = required_cols - set(raw_triples_df.columns)

print("missing:", missing)

assert not raw_triples_df.empty, "raw_triples_df đang rỗng"
assert not missing, f"Thiếu columns: {missing}"

shape: (136, 9)
columns: ['source_raw', 'source_type', 'relation', 'target_raw', 'target_type', 'source_chunk_id', 'published_date', 'evidence', 'confidence']
missing: set()


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [44]:
#@title 2.2 — Entity resolution

CORP_SUFFIXES = {
    "inc", "incorporated", "corp", "corporation",
    "ltd", "limited", "llc", "plc", "co", "company"
}

MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}


def norm_entity(name):
    s = unicodedata.normalize(
        "NFKC",
        norm_space(name)
    ).lower()

    s = re.sub(
        r"[^\w\s\-]",
        " ",
        s
    )

    return re.sub(
        r"\s+",
        " ",
        s
    ).strip()


def strip_suffix(name):
    toks = norm_entity(name).split()

    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()

    return " ".join(toks)


def merge_guard(a, b):
    na = strip_suffix(a)
    nb = strip_suffix(b)

    if not na or not nb:
        return False

    if na == nb:
        return True

    return SequenceMatcher(
        None,
        na,
        nb
    ).ratio() >= 0.72


EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None


def get_embedder():
    global embedder

    if embedder is None:
        embedder = SentenceTransformer(
            EMBED_MODEL
        )

    return embedder


class UF:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(
                self.p[x]
            )

        return self.p[x]

    def union(self, a, b):
        a = self.find(a)
        b = self.find(b)

        if a != b:
            self.p[b] = a


def build_resolution_map(
    raw_triples_df,
    threshold=0.90,
    top_k=5
):
    required = {
        "source_raw",
        "source_type",
        "target_raw",
        "target_type",
    }

    missing = required - set(
        raw_triples_df.columns
    )

    if missing:
        raise ValueError(
            f"raw_triples_df thiếu columns: {missing}"
        )

    if raw_triples_df.empty:
        raise ValueError(
            "raw_triples_df đang rỗng. "
            "Kiểm tra lại NER+RE extraction."
        )

    mentions = []

    for r in raw_triples_df.itertuples(
        index=False
    ):
        mentions += [
            (r.source_type, r.source_raw),
            (r.target_type, r.target_raw),
        ]

    counts = Counter(
        (t, norm_entity(n))
        for t, n in mentions
    )

    display_name = {}

    for t, n in mentions:
        display_name.setdefault(
            (t, norm_entity(n)),
            n
        )

    mapping = {}
    audit = []

    # Manual aliases
    for key in counts:
        t, norm = key

        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]

            audit.append({
                "type": t,
                "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0,
                "decision": "MERGE_MANUAL",
            })

    # Vector resolution per entity type
    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [
            k for k in counts
            if k[0] == typ
            and k not in mapping
        ]

        if not keys:
            continue

        names = [
            display_name[k]
            for k in keys
        ]

        vecs = get_embedder().encode(
            names,
            batch_size=128,
            show_progress_bar=False,
            normalize_embeddings=True,
        ).astype("float32")

        index = faiss.IndexFlatIP(
            vecs.shape[1]
        )

        index.add(vecs)

        sims, nbrs = index.search(
            vecs,
            min(top_k, len(names))
        )

        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(
                sims[i],
                nbrs[i]
            ):
                if (
                    j < 0
                    or i >= j
                    or float(score) < threshold
                ):
                    continue

                ok = merge_guard(
                    names[i],
                    names[j]
                )

                audit.append({
                    "type": typ,
                    "left": names[i],
                    "right": names[j],
                    "similarity": float(score),
                    "decision":
                        "MERGE_VECTOR"
                        if ok
                        else "REJECT_GUARD",
                })

                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)

        for i in range(len(names)):
            groups[
                uf.find(i)
            ].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (
                    -counts[keys[i]],
                    len(names[i]),
                    names[i].lower(),
                )
            )[0]

            canonical = names[best]

            for i in idxs:
                mapping[
                    keys[i]
                ] = canonical

    for key in counts:
        mapping.setdefault(
            key,
            display_name[key]
        )

    audit_df = pd.DataFrame(
        audit,
        columns=[
            "type",
            "left",
            "right",
            "similarity",
            "decision",
        ]
    )

    return mapping, audit_df


def canonicalize_triples(
    raw_df,
    mapping
):
    df = raw_df.copy()

    def canon(name, typ):
        n = norm_entity(name)

        return mapping.get(
            (typ, n),
            MANUAL_ALIASES.get(
                n,
                name
            )
        )

    df["source_name"] = [
        canon(n, t)
        for n, t in zip(
            df["source_raw"],
            df["source_type"]
        )
    ]

    df["target_name"] = [
        canon(n, t)
        for n, t in zip(
            df["target_raw"],
            df["target_type"]
        )
    ]

    df["source_name_norm"] = (
        df["source_name"].map(
            norm_entity
        )
    )

    df["target_name_norm"] = (
        df["target_name"].map(
            norm_entity
        )
    )

    df["source_id"] = [
        sha1(f"{t}:{n}")[:24]
        for t, n in zip(
            df["source_type"],
            df["source_name_norm"]
        )
    ]

    df["target_id"] = [
        sha1(f"{t}:{n}")[:24]
        for t, n in zip(
            df["target_type"],
            df["target_name_norm"]
        )
    ]

    return (
        df[
            df["source_id"]
            != df["target_id"]
        ]
        .reset_index(drop=True)
    )


entity_map, entity_resolution_audit_df = (
    build_resolution_map(
        raw_triples_df,
        threshold=0.90,
        top_k=5,
    )
)

triples_df = canonicalize_triples(
    raw_triples_df,
    entity_map
)

print("Raw triples:", len(raw_triples_df))
print("Canonical triples:", len(triples_df))
print("Audit rows:", len(entity_resolution_audit_df))

display(
    entity_resolution_audit_df.head(20)
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Raw triples: 136
Canonical triples: 136
Audit rows: 1


,type,left,right,similarity,decision
0,Technology,X90A,X90,0.920579,MERGE_VECTOR


In [ ]:
print(
    triples_df[
        [
            "source_name",
            "source_type",
            "relation",
            "target_name",
            "target_type",
        ]
    ].head(20)
)

                      source_name source_type        relation  \
0                      ServiceNow     Company  PARTNERED_WITH   
1                      ServiceNow     Company  PARTNERED_WITH   
2                          NVIDIA     Company  PARTNERED_WITH   
3  Integrated Water Services Inc.     Company        ACQUIRED   
4                          Chiron     Company            USES   

                           target_name target_type  
0                               NVIDIA     Company  
1                            Accenture     Company  
2                            Accenture     Company  
3  Hyperion International Technologies     Company  
4             cybersecurity technology  Technology  


In [45]:
#@title 2.3 — Node table + UNWIND bulk insert

def build_nodes(triples_df):
    rows = []

    for r in triples_df.itertuples(index=False):
        rows += [
            {
                "id": r.source_id,
                "name": r.source_name,
                "name_norm": r.source_name_norm,
                "type": r.source_type,
                "alias": r.source_raw,
            },
            {
                "id": r.target_id,
                "name": r.target_name,
                "name_norm": r.target_name_norm,
                "type": r.target_type,
                "alias": r.target_raw,
            },
        ]

    tmp = pd.DataFrame(rows)

    if tmp.empty:
        return pd.DataFrame(
            columns=[
                "id",
                "name",
                "name_norm",
                "type",
                "aliases",
                "aliases_norm",
            ]
        )

    out = []

    for (node_id, name, name_norm, typ), g in tmp.groupby(
        ["id", "name", "name_norm", "type"]
    ):
        aliases = sorted(
            set(
                g["alias"]
                .fillna("")
                .map(norm_space)
            )
        )

        aliases = [x for x in aliases if x]

        out.append(
            {
                "id": node_id,
                "name": name,
                "name_norm": name_norm,
                "type": typ,
                "aliases": aliases,
                "aliases_norm": sorted(
                    set(
                        norm_entity(x)
                        for x in aliases
                    )
                ),
            }
        )

    return pd.DataFrame(out)


def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i + size]


def validate_before_insert(nodes_df, triples_df):
    if nodes_df.empty:
        raise ValueError(
            "nodes_df đang rỗng. "
            "Kiểm tra lại Entity Resolution."
        )

    if triples_df.empty:
        raise ValueError(
            "triples_df đang rỗng. "
            "Kiểm tra lại NER+RE extraction."
        )

    required_cols = {
        "source_id",
        "target_id",
        "source_chunk_id",
        "published_date",
        "evidence",
        "confidence",
        "relation",
    }

    missing = required_cols - set(triples_df.columns)

    if missing:
        raise ValueError(
            f"triples_df thiếu columns: {missing}"
        )

    invalid = triples_df[
        triples_df["source_chunk_id"].isna()
        | triples_df["source_chunk_id"]
        .astype(str)
        .str.strip()
        .eq("")
        | triples_df["published_date"].isna()
        | triples_df["published_date"]
        .astype(str)
        .str.strip()
        .eq("")
    ]

    print("Prepared nodes:", len(nodes_df))
    print("Prepared edges:", len(triples_df))
    print(
        "Invalid provenance rows:",
        len(invalid)
    )

    if not invalid.empty:
        display(
            invalid[
                [
                    "source_name",
                    "relation",
                    "target_name",
                    "source_chunk_id",
                    "published_date",
                ]
            ].head(20)
        )

        raise ValueError(
            "Có edge thiếu source_chunk_id "
            "hoặc published_date."
        )

    invalid_relations = (
        set(triples_df["relation"].dropna())
        - set(ALLOWED_RELATIONS)
    )

    if invalid_relations:
        raise ValueError(
            f"Có relation ngoài allowlist: "
            f"{invalid_relations}"
        )

    print("✅ Pre-insert validation passed.")


def bulk_insert_nodes(
    nodes_df,
    batch_size=1000
):
    if nodes_df.empty:
        raise ValueError(
            "nodes_df đang rỗng."
        )

    inserted = 0

    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[
            nodes_df["type"] == typ
        ]

        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row

        MERGE (n:Entity {{id: row.id}})

        SET n:{typ},
            n.name = row.name,
            n.name_norm = row.name_norm,
            n.entity_type = row.type,
            n.aliases = row.aliases,
            n.aliases_norm = row.aliases_norm
        """

        records = part.to_dict(
            "records"
        )

        for b in batches(
            records,
            batch_size
        ):
            run_cypher(
                query,
                rows=b
            )

            inserted += len(b)

    print(
        f"✅ Node batches completed: "
        f"{inserted} records processed."
    )


def bulk_insert_edges(
    triples_df,
    batch_size=1000
):
    required = {
        "source_id",
        "target_id",
        "source_chunk_id",
        "published_date",
        "evidence",
        "confidence",
        "relation",
    }

    missing = required - set(
        triples_df.columns
    )

    if missing:
        raise ValueError(
            f"Missing edge fields: {missing}"
        )

    invalid = triples_df[
        triples_df["source_chunk_id"].isna()
        | triples_df["source_chunk_id"]
        .astype(str)
        .str.strip()
        .eq("")
        | triples_df["published_date"].isna()
        | triples_df["published_date"]
        .astype(str)
        .str.strip()
        .eq("")
    ]

    if not invalid.empty:
        raise ValueError(
            f"Có {len(invalid)} edge "
            "thiếu provenance."
        )

    inserted = 0

    cols = [
        "source_id",
        "target_id",
        "source_chunk_id",
        "published_date",
        "evidence",
        "confidence",
    ]

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[
            triples_df["relation"] == rel
        ]

        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row

        MATCH (
            s:Entity {{
                id: row.source_id
            }}
        )

        MATCH (
            t:Entity {{
                id: row.target_id
            }}
        )

        MERGE (
            s
        )-[
            r:{rel} {{
                source_chunk_id:
                    row.source_chunk_id
            }}
        ]->(
            t
        )

        SET r.published_date =
                row.published_date,
            r.evidence =
                row.evidence,
            r.confidence =
                row.confidence
        """

        records = (
            part[cols]
            .fillna("")
            .to_dict("records")
        )

        for b in batches(
            records,
            batch_size
        ):
            run_cypher(
                query,
                rows=b
            )

            inserted += len(b)

    print(
        f"✅ Edge batches completed: "
        f"{inserted} records processed."
    )


# =========================
# BUILD + VALIDATE
# =========================

nodes_df = build_nodes(
    triples_df
)

validate_before_insert(
    nodes_df,
    triples_df
)

display(
    nodes_df.head(20)
)


# =========================
# BULK INSERT
# =========================

bulk_insert_nodes(
    nodes_df,
    batch_size=1000
)

bulk_insert_edges(
    triples_df,
    batch_size=1000
)


# =========================
# POST-INSERT CHECKS
# =========================

node_count = run_cypher(
    """
    MATCH (n:Entity)
    RETURN count(n) AS nodes
    """
)

edge_count = run_cypher(
    """
    MATCH ()-[r]->()
    RETURN count(r) AS edges
    """
)

provenance_check = run_cypher(
    """
    MATCH ()-[r]->()

    WHERE
        r.source_chunk_id IS NULL
        OR trim(
            toString(
                r.source_chunk_id
            )
        ) = ''
        OR r.published_date IS NULL
        OR trim(
            toString(
                r.published_date
            )
        ) = ''

    RETURN
        count(r)
        AS invalid_provenance_edges
    """
)

print(
    "Neo4j node count:",
    node_count
)

print(
    "Neo4j edge count:",
    edge_count
)

print(
    "Provenance check:",
    provenance_check
)


# =========================
# RELATION DISTRIBUTION
# =========================

relation_stats = run_cypher(
    """
    MATCH ()-[r]->()

    RETURN
        type(r) AS relation,
        count(r) AS count

    ORDER BY count DESC
    """
)

display(
    pd.DataFrame(
        relation_stats
    )
)


# =========================
# SAMPLE EDGES
# =========================

sample_edges = run_cypher(
    """
    MATCH
        (s:Entity)-[r]->(t:Entity)

    RETURN
        s.name AS source,
        s.entity_type AS source_type,
        type(r) AS relation,
        t.name AS target,
        t.entity_type AS target_type,
        r.published_date AS published_date,
        r.source_chunk_id AS source_chunk_id,
        r.evidence AS evidence,
        r.confidence AS confidence

    LIMIT 20
    """
)

display(
    pd.DataFrame(
        sample_edges
    )
)

Prepared nodes: 222
Prepared edges: 136
Invalid provenance rows: 0
✅ Pre-insert validation passed.


,id,name,name_norm,type,aliases,aliases_norm
0,0087edc0e9756820936e0dc5,Samantha Rajapaksa,samantha rajapaksa,Person,[Samantha Rajapaksa],[samantha rajapaksa]
1,00ae3657cadb79d7d1810d3c,FSi Strategies,fsi strategies,Company,[FSi Strategies],[fsi strategies]
2,01b433b22d620a1f8f7803c5,Accenture,accenture,Company,[Accenture],[accenture]
3,01c0a1b8545f22f61ffe10fc,Squeeze,squeeze,Company,[Squeeze],[squeeze]
4,0430a89df77e6234ef375307,GreenPages,greenpages,Company,[GreenPages],[greenpages]
5,06d28138214cd51238fcc98c,X90,x90,Technology,"[X90, X90A]","[x90, x90a]"
6,07ee5c3f07cd4a5209aee14a,Jenne Cloud Services Brokerage,jenne cloud services brokerage,Company,[Jenne Cloud Services Brokerage],[jenne cloud services brokerage]
7,07f21a993871226abed12f9f,Senzary LLC,senzary llc,Company,[Senzary LLC],[senzary llc]
8,08193f3ca0efe2f71d179f4a,Renovus,renovus,Company,[Renovus],[renovus]
9,0c104aea0811fd35b52eaf70,General Services Administration,general services administration,Company,[General Services Administration],[general services administration]


✅ Node batches completed: 222 records processed.
✅ Edge batches completed: 136 records processed.
Neo4j node count: [{'nodes': 223}]
Neo4j edge count: [{'edges': 136}]
Provenance check: [{'invalid_provenance_edges': 0}]


,relation,count
0,USES,34
1,DEVELOPED,30
2,PARTNERED_WITH,27
3,WORKED_AT,14
4,ACQUIRED,13
5,LEADS,10
6,INVESTED_IN,4
7,FOUNDED,4


,source,source_type,relation,target,target_type,published_date,source_chunk_id,evidence,confidence
0,Accenture,Company,USES,AI Lighthouse,Technology,2023-07-27,a5a8b0ece135c638d4f0::c0000,ServiceNow ... announced the launch of AI Lighthouse,0.94
1,Integrated Water Services Inc.,Company,ACQUIRED,Hyperion International Technologies,Company,2023-03-07,104393b197a54876f491::c0000,"Integrated Water Services Inc. (""IWS"") ... recently acquired Hyperion International Technologies",1.00
2,NVIDIA,Company,PARTNERED_WITH,Accenture,Company,2023-07-27,a5a8b0ece135c638d4f0::c0000,ServiceNow NVIDIA and Accenture today announced the launch of AI Lighthouse,0.95
3,NVIDIA,Company,USES,AI Lighthouse,Technology,2023-07-27,a5a8b0ece135c638d4f0::c0000,ServiceNow ... announced the launch of AI Lighthouse,0.94
4,Chiron,Company,USES,cybersecurity technology,Technology,2023-02-27,a275f4f53d8d1a7d173e::c0000,Chiron's team ... employs cutting-edge cybersecurity technology,1.00
5,ServiceNow,Company,PARTNERED_WITH,Accenture,Company,2023-07-27,a5a8b0ece135c638d4f0::c0000,ServiceNow NVIDIA and Accenture today announced the launch of AI Lighthouse,0.95
6,ServiceNow,Company,PARTNERED_WITH,NVIDIA,Company,2023-07-27,a5a8b0ece135c638d4f0::c0000,ServiceNow NVIDIA and Accenture today announced the launch of AI Lighthouse,0.95
7,ServiceNow,Company,USES,generative AI,Technology,2023-05-18,8dbd50499a511691151c::c0000,ServiceNow ... capitalize on the recent rapid advancements seen in the field of generative AI,1.00
8,ServiceNow,Company,USES,large language models,Technology,2023-05-18,8dbd50499a511691151c::c0000,using large language models (LLMs),1.00
9,ServiceNow,Company,USES,AI Lighthouse,Technology,2023-07-27,a5a8b0ece135c638d4f0::c0000,ServiceNow ... announced the launch of AI Lighthouse,0.94


In [46]:
for v in [
    "chunks_df",
    "extraction_source",
    "raw_triples_df",
]:
    print(
        v,
        "✅ exists" if v in globals()
        else "❌ missing"
    )

chunks_df ✅ exists
extraction_source ✅ exists
raw_triples_df ✅ exists


In [47]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()

{'nodes': 223, 'edges': 136, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,cc9c6ee3857729e221d3f6de,ServiceNow,Company,6
1,23d7ceb58c062d83135817ba,Intelligent Technical Solutions,Company,3
2,1436a8eb4a4e8197fa650e48,Opsys Tech,Company,3
3,9960a1ca5dd13b4f18ca1fb7,New Charter Technologies,Company,3
4,8d3a4b002f6b4a95d8e04dc9,AI Lighthouse,Technology,3
5,70c4e949ff245096c6946802,Amazon Web Services,Company,3
6,b823fb02e58a12bdc4c38934,NVIDIA,Company,3
7,01b433b22d620a1f8f7803c5,Accenture,Company,3
8,483064d6337d5cd16bda1f38,PSG,Company,2
9,07f21a993871226abed12f9f,Senzary LLC,Company,2


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [48]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Flat vectors: 1500


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [49]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)

In [50]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [51]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
You are a text-only RAG answer generator.

Answer only from the supplied context.

Do not call tools.
Do not request external information.
Do not browse repositories.
Do not browse the web.
Do not output tool calls.
Do not output function calls.

Use only the supplied CONTEXT.

Be concise but complete.
Do not invent facts.

Cite provenance inline as [chunk_id=...]
whenever possible.

If evidence is insufficient or conflicting,
explicitly say that the supplied evidence
is insufficient or conflicting.
""".strip()

def generate_answer(question, context):
    prompt = f"""
QUESTION:
{question}

CONTEXT:
{context}

ANSWER:
""".strip()

    messages = [
        {
            "role": "system",
            "content": ANSWER_SYSTEM,
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    t0 = time.perf_counter()

    try:
        text, usage = groq_chat(
            messages,
            model=GROQ_MODEL,
            json_mode=False,
        )

    except Exception as e:
        msg = str(e)

        if (
            "tool_use_failed" not in msg
            and "Tool choice is none" not in msg
        ):
            raise

        print(
            "⚠️ Model attempted tool use. "
            "Retrying with stricter prompt."
        )

        strict_messages = [
            {
                "role": "system",
                "content": (
                    ANSWER_SYSTEM
                    + "\n\n"
                    "CRITICAL: You have no tools. "
                    "Never emit a tool call or function call. "
                    "Return normal prose only."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ]

        text, usage = groq_chat(
            strict_messages,
            model=GROQ_MODEL,
            json_mode=False,
        )

    return {
        "answer": text.strip(),
        "latency_s":
            time.perf_counter() - t0,
        "total_tokens":
            usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [52]:
#@title 4.1 — 5 câu Golden starter
GOLDEN_PATH = "/content/drive/MyDrive/graphrag_golden_50_first5000.csv"

golden_df = pd.read_csv(GOLDEN_PATH) if Path(GOLDEN_PATH).exists() else starter_golden.copy()
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")

,id,group,question,reference_answer,reference_evidence
0,G5000-01,multi-hop,Reconstruct the Aeris–Ericsson IoT transaction across the available reports: which Ericsson businesses moved to Aeri...,"Ericsson's IoT Accelerator and Connected Vehicle Cloud businesses, together with related assets, were to be transfer...",row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
1,G5000-02,cross-doc,"Did the first two Aeris/Ericsson reports describe a completed acquisition or a planned transfer, and what later evid...",The first reports describe a planned transaction: Aeris was to acquire Ericsson's IoT Accelerator and Connected Vehi...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
2,G5000-03,factoid,"After the Aeris–Ericsson IoT deal progressed, how many IoT devices, enterprises, and countries were cited in the lat...","More than 100 million IoT devices, 9,000 enterprises, and 190 countries.",row 935 (2023-01-18 22:37:00): A Leap in Connectivity: Aeris Acquires Technologies from Ericsson to Support Cellular...
3,G5000-04,cross-doc,"Which two named Ericsson IoT businesses recur across multiple reports of the Aeris transaction, and why should Graph...",The recurring businesses are Ericsson IoT Accelerator and Connected Vehicle Cloud. The reports describe the same Aer...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 1746 (2023-01-10 06:19:00): Aeris to...
4,G5000-05,multi-hop,"Starting from Ericsson, follow the graph to the acquirer and then to the reported IoT reach. What path and scale sho...",Ericsson -> (IoT Accelerator and Connected Vehicle Cloud transferred/acquired by) Aeris -> supports/connects more th...,row 33 (2022-12-07 13:45:00): Aeris to Acquire IoT Business from Ericsson | row 935 (2023-01-18 22:37:00): A Leap in...
5,G5000-06,multi-hop,Trace ServiceNow's generative-AI product/partner evolution from May through July 2023: what partnership began in May...,"In May, ServiceNow partnered with NVIDIA to build enterprise-grade generative AI for workflow automation. In June, S...",row 746 (2023-05-17 18:58:00): ServiceNow and NVIDIA Announce Partnership to Build Generative AI Across Enterprise I...
6,G5000-07,cross-doc,How did ServiceNow's July generative-AI expansion differ between its platform features and its ecosystem program wit...,ServiceNow expanded the Now Assist feature set with case summarization and text-to-code inside its workflow platform...,row 1435 (2023-07-26 20:30:00): ServiceNow Expands Generative AI Capabilities With Case Summarization and Text-to-Co...
7,G5000-08,multi-hop,"Which external organizations are connected to ServiceNow's generative-AI efforts in the selected data, and what dist...",NVIDIA is ServiceNow's generative-AI technology partner and later co-launch partner for AI Lighthouse; Accenture joi...,row 746 (2023-05-17 18:58:00): ServiceNow and NVIDIA Announce Partnership to Build Generative AI Across Enterprise I...
8,G5000-09,cross-doc,Two June 13 reports describe Now Assist for Virtual Agent. What should a deduplicated graph store as the core event ...,A single ServiceNow product event: Now Assist for Virtual Agent extends ServiceNow's generative-AI capabilities for ...,row 1742 (2023-06-13 13:50:00): ServiceNow Unveils Generative AI Solution With Real-Time Conversational Experiences ...
9,G5000-10,multi-hop,What sequence shows ServiceNow moving from domain-specific generative-AI exploration to concrete workflow features?,"At Knowledge 2023, ServiceNow emphasized domain-specific generative-AI/LLM use cases; in June it introduced Now Assi...",row 752 (2023-05-18 00:32:00): Knowledge 2023 - ServiceNow looks beyond general purpose generative AI to domain-spec...


In [125]:
#@title 4.2 — LLM-as-a-Judge (Structured Output strict)

from groq import Groq

judge_groq_client = Groq(api_key=GROQ_API_KEY)

JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "comprehensiveness": {
            "type": "integer",
            "minimum": 1,
            "maximum": 5
        },
        "faithfulness": {
            "type": "integer",
            "minimum": 1,
            "maximum": 5
        },
        "multi_hop_reasoning": {
            "type": "integer",
            "minimum": 1,
            "maximum": 5
        },
        "rationale": {
            "type": "string"
        }
    },
    "required": [
        "comprehensiveness",
        "faithfulness",
        "multi_hop_reasoning",
        "rationale"
    ],
    "additionalProperties": False
}


JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.

Evaluate the candidate answer using:
1. comprehensiveness: 1-5
2. faithfulness: 1-5
3. multi_hop_reasoning: 1-5

Use the reference answer as the correctness anchor.

Faithfulness must only consider claims supported by
the supplied candidate context.

Return the evaluation in the required schema.
""".strip()


def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":

        messages = [
            {
                "role": "system",
                "content": system
            },
            {
                "role": "user",
                "content": user
            }
        ]

        # ============================================
        # GPT-OSS: dùng Structured Output strict
        # ============================================
        if JUDGE_MODEL in {
            "openai/gpt-oss-20b",
            "openai/gpt-oss-120b",
        }:
            resp = judge_groq_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=messages,
                temperature=0,

                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "rag_judge_result",
                        "strict": True,
                        "schema": JUDGE_SCHEMA
                    }
                }
            )

            content = resp.choices[0].message.content

            if not content:
                raise RuntimeError(
                    "Judge trả về content rỗng."
                )

            return json.loads(content)

        # ============================================
        # QWEN: dùng JSON Object Mode
        # ============================================
        elif JUDGE_MODEL == "qwen/qwen3.6-27b":

            qwen_system = (
                system
                + "\n\n"
                "Return exactly ONE valid JSON object.\n"
                "Do not call tools.\n"
                "Do not use markdown.\n"
                "Do not use code fences.\n"
                "Do not output text outside the JSON object.\n"
                "Required keys:\n"
                "- comprehensiveness\n"
                "- faithfulness\n"
                "- multi_hop_reasoning\n"
                "- rationale"
            )

            try:
                resp = judge_groq_client.chat.completions.create(
                    model=JUDGE_MODEL,

                    messages=[
                        {
                            "role": "system",
                            "content": qwen_system
                        },
                        {
                            "role": "user",
                            "content": user
                        }
                    ],

                    temperature=0,

                    reasoning_effort="none",

                    response_format={
                        "type": "json_object"
                    }
                )

                content = (
                    resp.choices[0]
                    .message.content
                )

                if not content:
                    raise RuntimeError(
                        "Qwen Judge trả về content rỗng."
                    )

                return parse_json_object(content)

            except Exception as e:
                msg = str(e)

                print(
                    "⚠️ Qwen JSON mode failed:",
                    msg[:200]
                )

                # ----------------------------
                # fallback không ép JSON mode
                # ----------------------------
                resp = judge_groq_client.chat.completions.create(
                    model=JUDGE_MODEL,

                    messages=[
                        {
                            "role": "system",
                            "content": (
                                qwen_system
                                + "\nReturn JSON only."
                            )
                        },
                        {
                            "role": "user",
                            "content": user
                        }
                    ],

                    temperature=0,

                    reasoning_effort="none"
                )

                content = (
                    resp.choices[0]
                    .message.content
                )

                if not content:
                    raise RuntimeError(
                        "Qwen Judge fallback "
                        "trả về content rỗng."
                    )

                return parse_json_object(content)

        else:
            raise ValueError(
                f"Unsupported Groq judge model: "
                f"{JUDGE_MODEL}"
            )

    # ============================================
    # OPENAI
    # ============================================
    elif JUDGE_PROVIDER == "openai":

        if not OPENAI_API_KEY:
            raise RuntimeError(
                "Thiếu OPENAI_API_KEY."
            )

        from openai import OpenAI

        client = OpenAI(
            api_key=OPENAI_API_KEY
        )

        resp = client.chat.completions.create(
            model=JUDGE_MODEL,

            messages=[
                {
                    "role": "system",
                    "content": system
                },
                {
                    "role": "user",
                    "content": user
                }
            ],

            temperature=0,

            response_format={
                "type": "json_object"
            }
        )

        return parse_json_object(
            resp.choices[0].message.content
        )

    else:
        raise ValueError(
            "JUDGE_PROVIDER phải là "
            "'groq' hoặc 'openai'."
        )

def judge_answer(
    question,
    reference,
    answer,
    context
):
    # Cắt context để tránh token quá lớn
    context = str(context or "")[:6000]

    prompt = f"""
QUESTION:
{question}

REFERENCE ANSWER:
{reference}

CANDIDATE ANSWER:
{answer}

CANDIDATE CONTEXT:
{context}

Evaluate only the candidate answer.

Scoring:
- comprehensiveness:
  1 = very incomplete
  5 = fully addresses the question

- faithfulness:
  1 = mostly unsupported / hallucinated
  5 = fully supported by candidate context

- multi_hop_reasoning:
  1 = fails required relational reasoning
  5 = correctly performs all required reasoning

For simple factoid questions, score multi_hop_reasoning
based on whether the reasoning needed for that question
is correct; do not penalize merely because the question
is single-hop.
""".strip()

    obj = judge_json(
        JUDGE_SYSTEM,
        prompt
    )

    required = {
        "comprehensiveness",
        "faithfulness",
        "multi_hop_reasoning",
        "rationale",
    }

    missing = required - set(obj)

    if missing:
        raise ValueError(
            f"Judge output thiếu fields: {missing}"
        )

    # Defensive validation
    for key in [
        "comprehensiveness",
        "faithfulness",
        "multi_hop_reasoning",
    ]:
        score = int(obj[key])

        if not 1 <= score <= 5:
            raise ValueError(
                f"{key} ngoài khoảng 1-5: {score}"
            )

        obj[key] = score

    obj["rationale"] = str(
        obj["rationale"]
    ).strip()

    return obj

In [129]:
#@title 4.3 — Evaluation runner robust + checkpoint + error logging

from pathlib import Path
import pandas as pd
import traceback

CHECKPOINT = "/content/drive/MyDrive/graphrag_eval_checkpoint.csv"
ERROR_LOG = "/content/drive/MyDrive/graphrag_eval_errors.csv"

def run_evaluation(golden_df):
    # Resume checkpoint
    if Path(CHECKPOINT).exists():
        old_df = pd.read_csv(CHECKPOINT)
        rows = old_df.to_dict("records")
        done_ids = set(old_df["id"].astype(str))
        print(f"♻️ Resume: {len(done_ids)} câu đã hoàn thành")
    else:
        rows = []
        done_ids = set()
        print("🆕 Starting new evaluation")

    # Load errors cũ nếu có
    if Path(ERROR_LOG).exists():
        error_rows = pd.read_csv(ERROR_LOG).to_dict("records")
    else:
        error_rows = []

    remaining = golden_df[
        ~golden_df["id"].astype(str).isin(done_ids)
    ]

    print(f"Remaining: {len(remaining)}/{len(golden_df)}")

    for q in tqdm(
        remaining.itertuples(index=False),
        total=len(remaining),
        desc="Evaluation"
    ):
        print(f"\n▶ {q.id}: {q.question[:100]}")

        try:
            # -------------------
            # FLAT RAG
            # -------------------
            try:
                flat = answer_flat_rag(q.question)
            except Exception as e:
                raise RuntimeError(
                    f"[FLAT_RAG] {type(e).__name__}: {e}"
                ) from e

            # -------------------
            # GRAPH RAG
            # -------------------
            try:
                graph = answer_graph_rag(q.question)
            except Exception as e:
                raise RuntimeError(
                    f"[GRAPH_RAG] {type(e).__name__}: {e}"
                ) from e

            # -------------------
            # FLAT JUDGE
            # -------------------
            try:
                jf = judge_answer(
                    q.question,
                    q.reference_answer,
                    flat["answer"],
                    flat["context"]
                )
            except Exception as e:
                raise RuntimeError(
                    f"[FLAT_JUDGE] {type(e).__name__}: {e}"
                ) from e

            # -------------------
            # GRAPH JUDGE
            # -------------------
            try:
                jg = judge_answer(
                    q.question,
                    q.reference_answer,
                    graph["answer"],
                    graph["context"]
                )
            except Exception as e:
                raise RuntimeError(
                    f"[GRAPH_JUDGE] {type(e).__name__}: {e}"
                ) from e

            rows.append({
                "id": q.id,
                "group": q.group,
                "question": q.question,
                "reference_answer": q.reference_answer,

                "flat_answer": flat["answer"],
                "graph_answer": graph["answer"],

                "flat_comprehensiveness": jf["comprehensiveness"],
                "graph_comprehensiveness": jg["comprehensiveness"],

                "flat_faithfulness": jf["faithfulness"],
                "graph_faithfulness": jg["faithfulness"],

                "flat_multi_hop_reasoning": jf["multi_hop_reasoning"],
                "graph_multi_hop_reasoning": jg["multi_hop_reasoning"],

                "flat_latency_s": flat["latency_s"],
                "graph_latency_s": graph["latency_s"],

                "flat_total_tokens": flat.get("total_tokens"),
                "graph_total_tokens": graph.get("total_tokens"),

                "flat_judge_rationale": jf["rationale"],
                "graph_judge_rationale": jg["rationale"],

                "graph_supernode_events": len(
                    graph["graph_debug"]["diagnostics"]
                    .get("supernode_events", [])
                )
            })

            pd.DataFrame(rows).to_csv(
                CHECKPOINT,
                index=False
            )

            print(f"✅ {q.id} completed")

        except Exception as e:
            msg = str(e)

            error_rows.append({
            "id": q.id,
            "group": q.group,
            "question": q.question,
            "error": msg,
        })

            pd.DataFrame(error_rows).to_csv(
                ERROR_LOG,
                index=False
            )

            # RATE LIMIT -> DỪNG TOÀN BỘ
            if (
                "429" in msg
                or "rate_limit" in msg.lower()
                or "tokens per day" in msg.lower()
            ):
                print(f"\n⛔ Rate limit tại {q.id}")
                print("Checkpoint đã được giữ nguyên.")
                print(f"Completed: {len(rows)}")
                break

            # Các lỗi riêng lẻ khác -> skip câu đó
            print(f"\n⚠️ Skip {q.id}")
            print(msg)
            continue

    return pd.DataFrame(rows)

In [133]:
eval_results_df = run_evaluation(
    golden_df
)

print(
    "Completed:",
    len(eval_results_df),
    "/",
    len(golden_df)
)

♻️ Resume: 45 câu đã hoàn thành
Remaining: 5/50


Evaluation:   0%|          | 0/5 [00:00<?, ?it/s]


▶ G5000-46: Distinguish the cybersecurity relationships in the Keysight–Synopsys article and the LTTS–Palo Alto 
✅ G5000-46 completed

▶ G5000-47: In the Keysight–Synopsys IoT cybersecurity record, does the mention of Palo Alto Networks establish 
✅ G5000-47 completed

▶ G5000-48: Combine the three Snowflake-related records to summarize Snowflake's role in the 2023 cloud/data eco
✅ G5000-48 completed

▶ G5000-49: Across the selected Samsung records, identify three distinct technology domains Samsung is connected
✅ G5000-49 completed

▶ G5000-50: Compare the chip-related AI positioning of NVIDIA, AMD, and Intel in the selected 5,000-row scope. W
✅ G5000-50 completed
Completed: 50 / 50


In [134]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
eval_results_df.to_csv("/content/drive/MyDrive/graphrag_eval_results.csv", index=False)
comparison_df.to_csv("/content/drive/MyDrive/graphrag_vs_flatrag_summary.csv", index=False)

,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,3.136,3.455,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,4.818,4.955,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,3.364,3.682,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),7.597,6.555,GraphRAG không đắt hơn trong sample này.
4,cross-doc,Token usage,1725.045,1707.500,GraphRAG không đắt hơn trong sample này.
5,factoid,Comprehensiveness,2.000,2.600,Hai phương pháp gần nhau.
6,factoid,Faithfulness,5.000,5.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,2.600,3.200,Hai phương pháp gần nhau.
8,factoid,Latency (s),6.614,4.377,GraphRAG không đắt hơn trong sample này.
9,factoid,Token usage,1143.000,1077.800,GraphRAG không đắt hơn trong sample này.


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [135]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print("✅ Super-node cap OK.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()
show_resolution_audit(entity_resolution_audit_df)

{'id': 'cc9c6ee3857729e221d3f6de', 'name': 'ServiceNow', 'degree': 6} fetched= 6


,type,left,right,similarity,decision
0,Technology,X90A,X90,0.920579,MERGE_VECTOR


High-similarity rejected pairs:


,type,left,right,similarity,decision


## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [ ]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

community_df = build_communities()

In [ ]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau